# init

In [ ]:
import itertools
import os
from multiprocessing import Pool
from pathlib import Path

import h5py
import numpy as np
from numpy.linalg import LinAlgError

import gutpython

In [ ]:
initial_throwaway_time = 350
data_storage_dir = "/big-data/knappa/gutpython/sims"

In [ ]:
%matplotlib ipympl

In [ ]:
%load_ext jupyterlab_notify

# define param sampling

In [ ]:
default_numerical_param_dict = dict(
    max_stuck_chance=50,
    low_stuck_bound=2,
    unstuck_chance=10.0,
    mid_stuck_conc=10.0,
    seed_chance=5.0,
    seed_percent=5.0,
    absorption=0.0,
    reserve_fraction=0.0,
    bifido_lactate_production=0.005,
    flow_dist=0.28,
    bifido_doub=330,
    desulfo_doub=330,
    bacteroid_doub=330,
    clost_doub=330,
    # initialization constants
    init_num_bifidos=23562,
    init_num_bacteroids=5490,
    init_num_closts=921,
    init_num_desulfos=70,
)

In [ ]:
def numerical_param_sample():
    proportions = np.exp(np.log(2) * np.random.randn(len(default_numerical_param_dict)))
    return {
        param_name: (
            int(prop * param_value) if isinstance(param_value, int) else float(prop * param_value)
        )
        for prop, (param_name, param_value) in zip(
            proportions, default_numerical_param_dict.items()
        )
    }

In [ ]:
def control_param_sample():
    control_params = {}
    p = 0.1

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_bacteroids"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_bacteroids"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_bifidos"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_bifidos"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_closts"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_closts"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_desulfos"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_desulfos"] = lambda t: 0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["cs_inflow"] = lambda t: (
                0.2 if 0 <= t - initial_throwaway_time < 100 else 0.1
            )
        else:
            control_params["cs_inflow"] = lambda t: (
                0.05 if 0 <= t - initial_throwaway_time < 100 else 0.1
            )
    else:
        control_params["cs_inflow"] = lambda t: 0.1

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["fo_inflow"] = lambda t: (
                50.0 if 0 <= t - initial_throwaway_time < 100 else 25.0
            )
        else:
            control_params["fo_inflow"] = lambda t: (
                12.5 if 0 <= t - initial_throwaway_time < 100 else 25.0
            )
    else:
        control_params["fo_inflow"] = lambda t: 25.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["glucose_inflow"] = lambda t: (
                60.0 if 0 <= t - initial_throwaway_time < 100 else 30.0
            )
        else:
            control_params["glucose_inflow"] = lambda t: (
                15.0 if 0 <= t - initial_throwaway_time < 100 else 30.0
            )
    else:
        control_params["glucose_inflow"] = lambda t: 30.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["inulin_inflow"] = lambda t: (
                20.0 if 0 <= t - initial_throwaway_time < 100 else 10.0
            )
        else:
            control_params["inulin_inflow"] = lambda t: (
                5.0 if 0 <= t - initial_throwaway_time < 100 else 10.0
            )
    else:
        control_params["inulin_inflow"] = lambda t: 10.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["lactose_inflow"] = lambda t: (
                30.0 if 0 <= t - initial_throwaway_time < 100 else 15.0
            )
        else:
            control_params["lactose_inflow"] = lambda t: (
                7.5 if 0 <= t - initial_throwaway_time < 100 else 15.0
            )
    else:
        control_params["lactose_inflow"] = lambda t: 15.0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["lactate_inflow"] = lambda t: (
            1.0 if 0 <= t - initial_throwaway_time < 100 else 0.0
        )
    else:
        control_params["lactate_inflow"] = lambda t: 0.0

    return control_params

In [ ]:
def make_model(numerical_param_dict, control_param_dict):
    model = gutpython.GutPython(
        **numerical_param_dict,
        **control_param_dict,
        tick_in_flow=480,
    )
    model.setup()
    for t in range(initial_throwaway_time):
        model.go()

    return model

In [ ]:
def take_sample(sample_idx):
    model = make_model(numerical_param_sample(), control_param_sample())
    filename = os.path.join(data_storage_dir, f"sample-{str(sample_idx).zfill(4)}.hdf5")

    model.save(filename, write_mode="a")
    for t in range(initial_throwaway_time + 500):
        model.go()
        if t >= initial_throwaway_time:
            model.save(filename, write_mode="a")

# Sample/Simulate

In [ ]:
%%notify
with Pool(12) as p:
    p.map(take_sample, range(100))

# Parse 

In [ ]:
def parse(filename, time_seq):
    agent_vars = dict()
    computed_properties = dict()
    controls = dict()
    control_values = dict()
    init_parameters = dict()
    measurements = dict()
    molecules = dict()
    parameters = dict()
    with h5py.File(filename, "r") as h5file:
        keys = list(h5file[time_seq].keys())
        key_types = [h5file[time_seq][k].attrs["type"] for k in keys]
        GRID_WIDTH = int(h5file[time_seq]["GRID_WIDTH"][()])
        GRID_HEIGHT = int(h5file[time_seq]["GRID_HEIGHT"][()])

        for key, key_type in zip(keys, key_types):
            target_dict = {
                "agent": agent_vars,
                "computed_property": computed_properties,
                "control": controls,
                "control_value": control_values,
                "init_parameter": init_parameters,
                "measurement": measurements,
                "molecule": molecules,
                "parameter": parameters,
            }[key_type]
            target_dict[key] = h5file[time_seq][key][()]

    macrostate = np.array(
        [
            *[measurements[k] for k in sorted(measurements.keys())],
            *[parameters[k] for k in sorted(parameters.keys())],
            *np.concat(
                [np.atleast_1d(computed_properties[k]) for k in sorted(computed_properties.keys())],
                axis=0,
            ),
            *[control_values[k] for k in sorted(control_values.keys())],
        ],
        dtype=np.float32,
    )

    counts = np.zeros((GRID_WIDTH, GRID_HEIGHT, 4, 4), dtype=np.int64)
    age_energy_means = np.zeros((GRID_WIDTH, GRID_HEIGHT, 4, 4, 2), dtype=np.float32)
    cholesky = np.zeros((GRID_WIDTH, GRID_HEIGHT, 4, 4, 2 * (2 + 1) // 2), dtype=np.float32)
    for cell_type_idx, cell_type in enumerate(["bacteroid", "bifido", "clost", "desulfo"]):
        locs = agent_vars[f"{cell_type}_locations"].astype(np.int64)

        data = np.stack(
            (agent_vars[f"{cell_type}_age"], agent_vars[f"{cell_type}_energy"]), axis=-1
        )
        for seed_stuck_idx, (is_seed, is_stuck) in enumerate(
            itertools.product([False, True], repeat=2)
        ):
            mask = (agent_vars[f"{cell_type}_is_seed"] == is_seed) & (
                agent_vars[f"{cell_type}_is_stuck"] == is_stuck
            )
            count = np.sum(mask)
            if count > 0:
                for loc in locs[mask]:
                    counts[loc[0], loc[1], cell_type_idx, seed_stuck_idx] += 1
                for loc in itertools.product(range(GRID_WIDTH), range(GRID_HEIGHT)):
                    if counts[loc[0], loc[1], cell_type_idx, seed_stuck_idx] > 0:
                        loc_mask = mask & np.all(locs == loc, axis=1)
                        age_energy_means[loc[0], loc[1], cell_type_idx, seed_stuck_idx, :] = (
                            np.mean(data[loc_mask], axis=0)
                        )
                        if counts[loc[0], loc[1], cell_type_idx, seed_stuck_idx] > 1:
                            cov = np.cov(data[loc_mask].T)
                            # ensure that the covariance is positive definite by an adequate margin
                            cov += np.identity(2) * max(1e-6, (-1) * np.min(np.diag(cov)))

                            try:
                                cholesky[loc[0], loc[1], cell_type_idx, seed_stuck_idx, :] = (
                                    np.linalg.cholesky(cov, upper=False)[np.tril_indices(2)]
                                )
                            except LinAlgError:
                                # fallback is to assume no correlation, so that the Cholesky
                                # decomp is just based on the std-dev.
                                cholesky[loc[0], loc[1], cell_type_idx, seed_stuck_idx, :] = (
                                    np.array(
                                        [
                                            np.std(data[loc_mask][0], axis=0),
                                            0.0,
                                            np.std(data[loc_mask][1], axis=0),
                                        ]
                                    )
                                )
                        else:
                            # zero covariance when there is only one data point
                            cholesky[loc[0], loc[1], cell_type_idx, seed_stuck_idx, :] = np.array(
                                [0.0, 0.0, 0.0]
                            )

    agent_microstate = np.squeeze(
        np.concat(
            (
                counts[:, :, :, :, np.newaxis],
                age_energy_means,
                cholesky,
            ),
            axis=4,
        )
    )

    molecular_microstate = np.squeeze(
        np.array(
            [
                *[molecules[k] for k in sorted(molecules.keys())],
            ],
            dtype=np.float32,
        ).T
    )

    return agent_microstate, molecular_microstate, macrostate

In [ ]:
microstate_files = sorted(
    [
        f
        for f in Path(data_storage_dir).iterdir()
        if str(f.name).endswith(".hdf5") and str(f.name).startswith("sample-")
    ]
)

microstates = list()
for microstate_file in microstate_files:
    with h5py.File(microstate_file, "r") as h5file:
        for k in h5file.keys():
            microstates.append((microstate_file, k))

microstates = sorted(microstates)


def get_sample_number(file_name):
    file_name = str(file_name)
    start = file_name.rfind("-") + 1
    end = file_name.rfind(".")
    return int(file_name[start:end])

In [ ]:
def write_sample(x):
    microstate_filename, time_idx = x
    sample_number = get_sample_number(microstate_filename)

    agent_microstate, molecular_microstate, macrostate = parse(microstate_filename, time_idx)

    with h5py.File(
        os.path.join(
            microstate_dir,
            f"microstate-{str(sample_number).zfill(4)}-{str(time_idx).zfill(4)}.hdf5",
        ),
        "w",
    ) as h5file:
        h5file.create_dataset(
            "agent_microstate",
            shape=agent_microstate.shape,
            dtype=np.float32,
            data=agent_microstate,
            compression="gzip",
            compression_opts=9,
        )
        h5file.create_dataset(
            "molecular_microstate",
            shape=molecular_microstate.shape,
            dtype=np.float32,
            data=molecular_microstate,
            compression="gzip",
            compression_opts=9,
        )
        h5file.create_dataset(
            "macrostate",
            shape=macrostate.shape,
            dtype=np.float32,
            data=macrostate,
            compression="gzip",
            compression_opts=9,
        )

In [ ]:
microstate_dir = os.path.join(data_storage_dir, f"microstates")
try:
    os.mkdir(microstate_dir)
except FileExistsError:
    pass

In [ ]:
%%notify
with Pool(12) as p:
    p.map(write_sample, microstates)